# pytempo: un tur ghidat

`pytempo` citeste statistica oficiala romaneasca din API-ul INS TEMPO Online:
gaseste indicatori, le citeste metadatele, aduce datele intr-un DataFrame
pandas, le reasaza si scrie SQL-ul cu care le incarci in PostgreSQL.

**Acest notebook ruleaza live pe serverul INS.** Fiecare celula face cereri
reale, deci are nevoie de conexiune, iar cateva celule dureaza cateva
secunde. Exemplele au fost alese sa ramana mici si politicoase: nicio celula
de aici nu face mai mult de cateva cereri.

Se instaleaza direct din GitHub:

    pip install git+https://github.com/CIDS-UBB/pytempo.git


In [1]:
import pytempo as t

## 1. Descoperire

Incepe cu o panorama: cat de mare e catalogul si de unde se porneste.


In [2]:
t.overview()

pytempo: 1916 TEMPO indicators, in 8 top level domains.
Start with find('salariati') or domains(). t.help() has the full guide.


`find` e cautarea simpla pe cuvinte. Se uita in numele indicatorului si in
cod, ignora diacriticele si majusculele, si cere ca **toate** cuvintele date
sa se potriveasca. Intoarce **toate** potrivirile, nu o pagina taiata, deci
taie rezultatul daca vrei doar cateva.


In [3]:
results = t.find("salariati")
print(len(results), "indicators")
results[:5]

104 indicators


[Matrix('AMG1103', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe grupe de varsta si sexe'),
 Matrix('AMG1104', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe grupe de varsta si medii de rezidenta'),
 Matrix('AMG1105', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe activitati si sexe'),
 Matrix('AMG1106', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe activitati si medii de rezidenta'),
 Matrix('AMG115K', 'AMIGO - Salariati cu regim de lucru temporar dupa durata obisnuita a saptamanii de lucru si sexe')]

`search` e cealalta unealta: descoperire cu filtre. Cuvantul cautat e
optional, iar filtrele se combina intre ele.

* `domeniu` se potriveste pe un subsir din numele domeniului statistic, deci
  `economic` gaseste `B. STATISTICA ECONOMICA` fara sa stii forma exacta.
* `periodicitate` se potriveste pe un subsir din cat de des se publica.
* `level` pastreaza doar indicatorii care ajung la acel nivel teritorial.
* `caen=True` pastreaza doar pe cei cu o clasificare de activitati CAEN.

Diferenta intr-o linie: **`find` cauta in nume, `search` filtreaza pe
metadate.**


In [4]:
economic = t.search(domeniu="economic", periodicitate="anuala", level="judet")
print(len(economic), "indicators")
economic[:5]

111 indicators


[Matrix('AGR101A', 'Suprafata fondului funciar dupa modul de folosinta, pe forme de proprietate, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR101B', 'Suprafata fondului funciar dupa modul de folosinta, pe judete si localitati'),
 Matrix('AGR102A', 'Suprafata terenurilor amenajate cu lucrari de irigatii si suprafata agricola irigata, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR102B', 'Suprafata terenurilor amenajate cu lucrari de desecare, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR102C', 'Suprafata terenurilor amenajate cu lucrari de ameliorare si combaterea eroziunii solului, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete')]

`t.filters()` afiseaza pe ce se poate filtra, cu valorile reale citite din
catalog, nu dintr-o lista scrisa de mana.


In [5]:
t.filters()

Filters for t.search(). They combine with each other and with the search words.
  level        : ['national', 'macroregiune', 'regiune', 'judet', 'localitate', 'necunoscut']
  caen         : True only those with a CAEN dimension, False only those without
  domeniu      : substring of the domain name, diacritics ignored
                 A. STATISTICA SOCIALA
                 B. STATISTICA ECONOMICA
                 C. FINANTE
                 D. JUSTITIE
                 E. MEDIU INCONJURATOR
                 F. UTILITATI PUBLICE SI ADMINISTRAREA TERITORIULUI
                 G. DEZVOLTARE DURABILA - Orizont 2020
                 H. DEZVOLTARE DURABILA - Tinte 2030
  periodicitate: substring of the periodicity, diacritics ignored
                 ['5 - 6 ani', 'Anuala', 'Cincinal', 'La 2 ani', 'La fiecare 3 ani sau mai mult', 'La trei ani', 'Lunara', 'Perioada neregulata', 'Recomandabil la 10 ani', 'Trimestriala']

The metadata filters rest on the local index. If it is missing,
search a

## 2. Intelegerea unui indicator

Inainte sa tragi date, citeste ce este de fapt indicatorul. Folosim FOM104D,
numarul mediu al salariatilor pe judete si localitati.


In [6]:
m = t.matrix("FOM104D")
m

Matrix('FOM104D', 'Numarul mediu al salariatilor pe judete si localitati')

`what()` e versiunea scurta: prima fraza din definitie, unitatea de masura,
cat de des se publica, cand a fost actualizat ultima data, si un avertisment
daca observatiile mentioneaza ani anume, ceea ce de obicei semnaleaza o
ruptura de serie.


In [7]:
m.what()

FOM104D  Numarul mediu al salariatilor pe judete si localitati
  Numarul mediu al salariatilor cuprinde persoanele angajate cu contract de munca/raport de serviciu pe durata determinata sau nedeterminata (inclusiv lucratorii sezonieri, managerul sau administratorul), al caror contract de munca/raport de serviciu nu a fost suspendat in perioada de referinta.
  unit        : Numar persoane
  periodicity : Anuala
  updated     : 20-11-2025
                read them with .describe()


`where()` arata unde sta indicatorul in arborele de domenii si ce acopera:
cate unitati teritoriale pe fiecare nivel, daca localitatile poarta cod
SIRUTA, si intervalul de ani.

O nota despre **nivelele teritoriale**. `national`, `macroregiune`,
`regiune`, `judet`, `localitate` sunt felul in care pytempo interpreteaza
denumirile optiunilor, nu un concept expus direct de INS. O dimensiune
teritoriala amesteca de obicei toate nivelele intr-o singura coloana, iar
pytempo deduce care e care, ca sa poti cere unul anume. Denumirile care nu
se incadreaza in nomenclatorul administrativ, cum sunt statiile de
monitorizare, primesc `necunoscut` in loc sa fie fortate intr-un nivel.


In [8]:
m.where()

domain   : A. STATISTICA SOCIALA > FORTA DE MUNCA > SALARIATI
territory: Judete (43 options)
    national        1
    judet           42
territory: Localitati (3183 options)
    localitate      3182
SIRUTA   : yes
time     : Ani, 35 periods, 1990 to 2024


`how()` genereaza manualul de descarcare pentru acest indicator anume:
comenzile care au sens pentru el, strategia care se va folosi, si la cate
cereri sa te astepti.

Spune si pe care dintre cele doua cai de descarcare trebuie luat indicatorul
asta, `get()` sau `download()`, si tipareste comanda gata de copiat.


In [9]:
m.how()

How to download FOM104D:
  m = t.matrix('FOM104D')
  df = m.get()          level localitate, tidied
  m.get(raw=True)       exactly what INS returns, no extras
  m.download(folder='data/fom104d')
                        the same data, written straight to a CSV on disk

  TERRITORIAL LEVEL, what level= takes here:
    national          1 unit,     1 request   m.get(level='national')
    judet           42 units,     1 request   m.get(level='judet')
    localitate    3182 units,   42 requests   m.get(level='localitate')   the finest, and the default
    every level at once                       m.get(level=None)

  county and locality are separate dimensions here, so a level picks
  which one is active and puts the other on TOTAL: level='judet' gives
  one row per county in a single request, level='localitate' gives the
  localities, county by county.

  FILTERS: none to add. This indicator is territory and time only, so
  level= is the whole choice.

  A TYPICAL CALL for this indicator:

`describe()` afiseaza fisa integrala, exact cum a scris-o INS: definitia
completa, metodologia, sursele si observatiile. E lunga intentionat.
Observatiile sunt locul unde stau rupturile de serie si avertismentele
despre ani incompleti, deci citeste-le inainte sa te bazezi pe o serie.


In [10]:
m.describe()

FOM104D  Numarul mediu al salariatilor pe judete si localitati
domain      : A. STATISTICA SOCIALA > FORTA DE MUNCA > SALARIATI
levels      : national, judet, localitate
periodicity : Anuala
updated     : 20-11-2025

DEFINITION
Numarul mediu al salariatilor cuprinde persoanele angajate cu contract de munca/raport de serviciu pe durata determinata sau nedeterminata (inclusiv lucratorii sezonieri, managerul sau administratorul), al caror contract de munca/raport de serviciu nu a fost suspendat in perioada de referinta.
Numarul mediu al salariatilor se calculeaza ca medie aritmetica simpla rezultata din suma efectivelor zilnice de salariati (exclusiv cei al caror contract de munca/raport de serviciu a fost suspendat), din perioada de referinta, inclusiv din zilele de repaus saptamanal, sarbatori legale si alte zile nelucratoare, impartita la numarul total al zilelor calendaristice.
In efectivul zilnic al salariatilor luat in calculul numarului mediu se cuprind urmatoarele categorii:
- sal

`options()` fara argument listeaza dimensiunile, fiecare cu rolul pe care
i l-a dat pytempo si cu cate valori are. Cu un argument listeaza valorile
unei dimensiuni, numita dupa label, rol, index sau nivel.


In [11]:
m.options()

[0] Judete (teritoriu/judet, 43 options)
[1] Localitati (teritoriu/localitate, 3183 options)
[2] Ani (timp, 35 options)
[3] UM: Numar persoane (um, 1 options)

Rolurile poarta un al doilea semn pentru teritoriu: `teritoriu/judet` si
`teritoriu/localitate` de mai sus deosebesc o dimensiune de judete de una de
localitati fara sa citesti eticheta. Conteaza, fiindca eticheta nu e un ghid de
incredere. FOM104D le zice localitatilor `Localitati`; GOS102A le zice
`Municipii si orase`; coloanele derivate urmeaza ce a scris INS, fiindca
biblioteca nu redenumeste nimic.

Deci codul din aval intreaba, in loc sa se potriveasca pe o litera.
`territory_columns()` da numele coloanelor teritoriului celui mai fin, pe chei
dupa ce contin, gata de rename:


In [29]:
print("teritoriul fin este:", m.locality_dimension.label.strip())
m.territory_columns()


teritoriul fin este: Localitati


{'label': 'Localitati',
 'siruta': 'Localitati_siruta',
 'nivel': 'Localitati_nivel',
 'tip': 'Localitati_tip',
 'nume': 'Localitati_nume'}

## 3. O extragere simpla

FOM101A, resursele de munca pe judete, incape intr-o singura cerere, deci e
o prima tragere buna.

`get()` fara argumente face trei lucruri implicit: alege cel mai fin nivel
teritorial pe care il atinge indicatorul, aplica standardizarea tidy, si
afiseaza o linie cu ce a hotarat. Cand implicitul lasa pe dinafara nivelele
mai grosiere, le numeste si iti spune cum le recuperezi.


In [12]:
df = t.matrix("FOM101A").get()
df.shape

FOM101A: level judet (the finest), single, 1 request
  for every level, including national, macroregiune and regiune, use get(level=None)


(4392, 7)

Rezultatul e in **format lung**: un rand per combinatie, o coloana text per
dimensiune, o coloana numerica `Valoare`, si apoi coloanele derivate adaugate
de tidy.

Tidy adauga doar coloanele care chiar contin ceva. O dimensiune de judete nu
are cod SIRUTA si nici tip de asezare, deci primeste doar coloana de nivel.


In [13]:
df.head()

,Sexe,"Macroregiuni, regiuni de dezvoltare si judete",Ani,UM: Mii persoane,Valoare,"Macroregiuni, regiuni de dezvoltare si judete_nivel",Ani_an
0,Total,Arges,Anul 1990,Mii persoane,394.8,judet,1990
1,Total,Arges,Anul 1991,Mii persoane,394.7,judet,1991
2,Total,Arges,Anul 1992,Mii persoane,399.5,judet,1992
3,Total,Arges,Anul 1993,Mii persoane,392.9,judet,1993
4,Total,Arges,Anul 1994,Mii persoane,392.2,judet,1994


Coloanele derivate sunt tipizate, nu text: `Int64` pentru an, string
nullable pentru nivel.


In [14]:
df.dtypes

Sexe                                                       str
Macroregiuni, regiuni de dezvoltare si judete              str
Ani                                                        str
UM: Mii persoane                                           str
Valoare                                                float64
Macroregiuni, regiuni de dezvoltare si judete_nivel     string
Ani_an                                                   Int64
dtype: object

Cererea unui nivel schimba ce vine inapoi. Judetele si regiunile sunt felii
diferite din acelasi indicator, deci numarul de randuri difera.


In [15]:
counties = t.matrix("FOM101A").get(level="judet", progress=False)
regions = t.matrix("FOM101A").get(level="regiune", progress=False)
print("judet  :", counties.shape)
print("regiune:", regions.shape)

judet  : (4392, 7)
regiune: (840, 7)


`raw=True` da exact ce a intors INS, fara coloane derivate. Foloseste brut
cand vrei sa vezi sursa neatinsa sau iti scrii propria prelucrare; foloseste
tidy, care e implicit, cand vrei sa lucrezi cu datele.


In [16]:
raw = t.matrix("FOM101A").get(raw=True, progress=False)
print("raw :", raw.shape)
print("tidy:", df.shape)
raw.head(3)

raw : (4392, 5)
tidy: (4392, 7)


,Sexe,"Macroregiuni, regiuni de dezvoltare si judete",Ani,UM: Mii persoane,Valoare
0,Total,Arges,Anul 1990,Mii persoane,394.8
1,Total,Arges,Anul 1991,Mii persoane,394.7
2,Total,Arges,Anul 1992,Mii persoane,399.5


## 4. Standardizarea si SIRUTA

SIRUTA e codul oficial al unei unitati administrative din Romania. INS il
pune in interiorul denumirii localitatii, ca prefix numeric, deci eticheta
bruta arata asa: `1017 MUNICIPIUL ALBA IULIA`.

pytempo il desface in coloane separate si **pastreaza denumirea originala
neatinsa**. SIRUTA se pastreaza ca si CHEIE, niciodata nu se arunca, fiindca
el e cel care iti permite sa legi datele astea de alte surse administrative:
registre de populatie, bugete, geografii.

Folosim aici SAN103B, copiii inscrisi in crese pe judete si localitati,
fiindca ajunge la nivel de localitate si totusi incape intr-o singura cerere.


In [17]:
nurseries = t.matrix("SAN103B").get()
localities = nurseries[nurseries["Localitati_nivel"] == "localitate"]
localities[["Localitati", "Localitati_siruta", "Localitati_tip",
            "Localitati_nume", "Valoare"]].head(6)

SAN103B: all levels, single, 1 request


,Localitati,Localitati_siruta,Localitati_tip,Localitati_nume,Valoare
2,1017 MUNICIPIUL ALBA IULIA,1017,municipiu,ALBA IULIA,50
3,1213 MUNICIPIUL AIUD,1213,municipiu,AIUD,41
5,9262 MUNICIPIUL ARAD,9262,municipiu,ARAD,159
6,9459 ORAS CHISINEU-CRIS,9459,oras,CHISINEU-CRIS,14
7,9538 ORAS INEU,9538,oras,INEU,18
8,9574 ORAS LIPOVA,9574,oras,LIPOVA,50


Observa ce s-a intamplat: codul, tipul de asezare si denumirea curata sunt
acum trei coloane separate si tipizate, in timp ce coloana `Localitati`
pastreaza textul original.

Inca un lucru de stiut: **datele sunt rare.** Combinatiile fara date lipsesc
ca randuri intregi, nu vin ca `NaN`. Asta reflecta istorie administrativa
reala, nu o lipsa a bibliotecii: Ilfov si Municipiul Bucuresti nu exista ca
unitati separate inainte de 1996, deci acele randuri pur si simplu nu exista.
Nu verifica o descarcare comparand numarul de randuri cu produsul
dimensiunilor.


In [18]:
print("rows returned :", len(nurseries))
print("empty values  :", int(nurseries["Valoare"].isna().sum()))

rows returned : 180
empty values  : 0


Zeroul ala merita o oprire. INS scrie `:` cand nu are cifra si `0` cand a
masurat zero, iar astea sunt doua afirmatii diferite. pytempo le tine separate
in singurul mod cinstit: un `:` vine ca **rand absent**, iar un `0` ca rand cu
`Valoare` egala cu `0.0`.

Deci cadrul nu e o grila completa, iar un an care lipseste pentru o comuna e
necunoscut, nu zero. Ce faci cu golurile e decizia ta, iar `df.tempo.coverage()`,
mai jos, arata unde sunt.


## 5. Doua dimensiuni teritoriale

Unii indicatori pastreaza judetul si localitatea ca **doua dimensiuni
separate**. FOM104D e unul: are `Judete` cu 43 de optiuni si `Localitati` cu
3183.

Acolo, nivelul alege ce dimensiune e activa si o pune pe cealalta pe totalul
ei. `level='judet'` da un rand per judet, cu localitatile fixate pe TOTAL,
intr-o singura cerere. `level='localitate'` da localitatile, iar fiindca
datele sunt cheiate pe perechea reala judet plus localitate, dimensiunea de
judete ramane intreaga si descarcarea merge judet cu judet.

Rulam aici varianta ieftina.


In [19]:
by_county = t.matrix("FOM104D").get(level="judet")
print(by_county.shape)
by_county.head(3)

FOM104D: level judet, single, 1 request
  for every level, including national and localitate, use get(level=None)


(1469, 8)


,Judete,Localitati,Ani,UM: Numar persoane,Valoare,Judete_nivel,Localitati_nivel,Ani_an
0,Alba,TOTAL,Anul 1990,Numar persoane,149181,judet,national,1990
1,Alba,TOTAL,Anul 1991,Numar persoane,137228,judet,national,1991
2,Alba,TOTAL,Anul 1992,Numar persoane,129564,judet,national,1992


## 6. Filtrare: doar optiunile care te intereseaza

`select=` ingusteaza o dimensiune inainte sa se construiasca interogarea, deci
nu se descarca nimic din ce nu ai cerut. Primeste o lista de etichete, o lista
de nomItemId, un predicat, sau un cuvant.

Cuvintele sunt pentru dimensiunile care au niveluri in interior. POP107D are
104 varste: `Total`, optsprezece grupe cincinale si cele 85 de varste
individuale de sub ele. Ca sa ceri grupele scriai pana acum o bucla peste
etichete, pastrandu-le pe cele cu cratima sau pe cea care zice Total, adica o
presupunere despre cum scrie INS denumirile, care se strica la dimensiunea
urmatoare. Acum e un cuvant, iar `m.options()` arata ce ar pastra inainte sa se
descarce ceva:


In [30]:
p = t.matrix("POP107D")
p.options("varsta", kind="groups")


Total,    0- 4 ani,    5- 9 ani,    10-14 ani,    15-19 ani,    20-24 ani,    25-29 ani,    30-34 ani,    35-39 ani,    40-44 ani,    45-49 ani,    50-54 ani,    55-59 ani,    60-64 ani,    65-69 ani,    70-74 ani,    75-79 ani,    80-84 ani,    85 ani si peste

`kind` primeste `groups`, sau `parents`, pentru agregate, `leaves` pentru
nivelul cel mai fin si `total` pentru total. Merge la orice dimensiune cu
niveluri, nu doar la varste: la AGR101A, modul de folosinta a fondului funciar,
`groups` da cele patru categorii agregate si `leaves` cele zece de sub ele.

De unde vin nivelurile a fost masurat, nu presupus. `parentId` e null pe orice
dimensiune in afara de localitati, unde arata spre judet, adica spre cu totul
alta dimensiune; `offset` e o simpla ordine. Ce are INS cu adevarat e indentarea
etichetei, trei spatii pe nivel, din care isi deseneaza propriul arbore, deci
aia e rezerva.

Acelasi cuvant intra in `select=`, iar tot ce urmeaza lucreaza pe setul redus:
numarul de celule, spargerea in bucati si interogarea insasi. Nouasprezece
varste in loc de o suta patru e diferenta dintre o cerere si mai multe:


In [ ]:
groups = p.get(level="judet", select={"varsta": "groups"})
print(groups.shape)
groups.head(3)


## 7. O descarcare mai mare, si download() pentru cele mari

**Aceasta celula dureaza mai mult decat celelalte.** Face mai multe cereri.

Un singur POST catre INS e limitat la un buget de celule. Cand un indicator e
mai mare de atat, pytempo sparge lucrul automat si concateneaza bucatile:
judet cu judet pentru indicatorii care ajung la nivel de localitate, altfel
pe cea mai mare dimensiune. Nu trebuie sa planifici nimic, dar e bine sa stii
ca se intampla, si de aceea `get()` afiseaza linia de decizie.

FOM106E se sparge pe dimensiunea CAEN in cateva cereri. Indicatorii care ar
cere sute de cereri nu sunt folositi in acest tutorial: peste 50 de cereri
`get()` se opreste si te trimite la `download()`, care e subiectul celulei
urmatoare.


In [20]:
big = t.matrix("FOM106E").get()
big.shape

FOM106E: level judet (the finest), split:CAEN Rev.2  (activitati ale economiei nationale), 2 requests
  for every level, including national, macroregiune and regiune, use get(level=None)


  1/2: +84354 rows (total 84354)


  2/2: +45256 rows (total 129610)


(129610, 8)

### Cand spargerea nu ajunge: download()

FOM106E a fost cateva cereri. Unii indicatori sunt sute: POP107D la nivel de
localitate are 380, si `get()` refuza sa il porneasca. Nu din prudenta. `get()`
tine fiecare cerere in memorie pana vine ultima, deci un singur timeout tarziu,
si INS chiar da timeout, pierde toata descarcarea si nu are de unde relua.
Masurat pe SAN101B: cinci ore prin `get()` si abandonat, sub trei minute
scriind fiecare judet pe disc.

`download()` e aceeasi cerere, pentru cazul asta. Aceleasi `level`, `levels` si
`select`, acelasi plan, construit de acelasi cod. Difera doar unde ajung
raspunsurile: fiecare cerere se scrie in felia ei pe disc imediat ce vine. Aici
e pe ceva destul de mic pentru un tutorial, GOS102A, a carui dimensiune de
localitati se cheama `Municipii si orase`:


In [ ]:
orase = t.matrix("GOS102A").download(level="judet", folder="data/gos102a")
orase.shape


Memoria sta la o cerere, o rulare intrerupta pastreaza ce a apucat, iar aceeasi
comanda rulata din nou cere doar feliile care nu sunt pe disc. O cerere care tot
cade e raportata la final in loc sa darame restul.

Verificarea de agregare de la final e ultima linie: cate felii au ajuns fata de
cate s-au planificat, randurile cadrului lipit fata de suma randurilor
feliilor, nicio combinatie de dimensiuni de doua ori si, cand s-a dat un
`select`, fiecare dimensiune filtrata inapoi la marimea ceruta. Orice
neregula se tipareste, nu se lasa in fisier, iar cadrul poarta acelasi verdict
in `df.attrs['complete']` si `df.attrs['aggregation_warnings']`.

`return_df=False` intoarce calea CSV-ului in loc de cadru, pentru indicatorii
prea mari ca sa incapa in memorie.

Si uite ce se intampla daca ceri unul dintre aceia cu `get()`. Se opreste
inainte sa trimita ceva, tipareste ce ai de facut si ridica un
`MatrixTooLargeError` de o singura linie, care e un `ValueError`, deci codul
care prinde deja ValueError merge neatins:


In [31]:
try:
    t.matrix("POP107D").get()
except t.MatrixTooLargeError as e:
    print("oprit:", e)


POP107D: all levels, by_county, 380 requests

POP107D IS TOO LARGE FOR get(). Nothing has been downloaded.
  380 requests, over the 50 get() will hold in memory. get() keeps
  every one of them until the last comes back, so a single late timeout,
  and INS does time out, loses all of it with nothing to resume from.

  Use download() instead, which is the same call through disk:
    m.download(folder='data/pop107d')
  It writes each request as it arrives, resumes where it stopped, and
  retries on timeout.

  m.how()                the whole menu for POP107D: every level, every filter
  m.get(confirm=False)   go ahead with get() anyway, in memory, no checkpoint

oprit: POP107D: 380 requests, use download(). See the guidance above, or m.how().


## 8. Reasezarea: accessorul df.tempo

Orice cadru venit din `get(tidy=True)` are un accessor `df.tempo`. El
reaseaza si rezuma, nu aduce nimic de pe retea, si nu modifica cadrul primit.
Mai jos nu se face nicio cerere in plus: refolosim cadrul FOM101A.

`coverage()` e prima privire pe o serie: un rand per unitate teritoriala,
intervalul de ani pe care il are, cati dintre anii vazuti oriunde in cadru ii
lipsesc, si valoarea minima si maxima cu anul in care a aparut fiecare.


In [21]:
df.tempo.coverage().head(8)

,"Macroregiuni, regiuni de dezvoltare si judete",first_year,last_year,n_years,missing_years,min_value,min_year,max_value,max_year
0,Alba,1990,2024,35,0,92.1,2019,246.1,1990
1,Arad,1990,2024,35,0,123.1,2022,299.3,2011
2,Arges,1990,2024,35,0,168.4,2022,412.5,2011
3,Bacau,1990,2024,35,0,165.5,2019,473.7,2010
4,Bihor,1990,2024,35,0,168.0,2022,391.4,1990
5,Bistrita-Nasaud,1990,2024,35,0,80.2,2019,209.7,2010
6,Botosani,1990,2024,35,0,105.4,2019,280.0,2010
7,Braila,1990,2024,35,0,78.4,2022,238.9,2005


`wide()` pivoteaza timpul in coloane, forma pe care ai pune-o intr-o
lucrare. Indexul se construieste din coloanele originale de dimensiune, fara
cele derivate, fara coloana originala de timp, care spune acelasi lucru ca
anul, si fara o coloana de unitate de masura care nu variaza.


In [22]:
wide = df.tempo.wide()
print(wide.shape)
wide.iloc[:4, :8]

(129, 37)


,Sexe,"Macroregiuni, regiuni de dezvoltare si judete",1990,1991,1992,1993,1994,1995
0,Feminin,Alba,116.5,113.0,114.5,116.8,117.2,113.1
1,Feminin,Arad,140.8,135.2,133.5,131.7,129.5,134.8
2,Feminin,Arges,191.4,191.2,200.3,196.5,195.8,194.0
3,Feminin,Bacau,197.9,194.2,198.1,198.6,202.1,212.0


`df.tempo.geo()` e o schita documentata. Va face join pe SIRUTA si va
intoarce un GeoDataFrame, si va veni ca extra optional `pytempo[geo]`, ca
cine vrea doar cifrele sa nu plateasca pentru stiva de geometrii.

Apelat pe un cadru care nu e iesire tidy, accessorul o spune limpede, in loc
sa ghiceasca.


In [23]:
try:
    df.tempo.geo()
except NotImplementedError as e:
    print(e)

geo() is not implemented yet: it will join on SIRUTA and return a GeoDataFrame, arriving as the optional pytempo[geo] extra


Inca un lucru pe care il face accessorul, si e singura verificare care conteaza
pana la urma: `spot_check()` pregateste comparatia cu site-ul TEMPO Online.
Alege unitati la intamplare, fixeaza fiecare alta dimensiune pe totalul ei ca
site-ul sa arate o singura serie, spune pe ce a fixat si tipareste seria an cu
an. `seed=` face alegerea reproductibila, deci poate sta intr-un script. Nu
atinge reteaua: citeste cadrul pe care il ai deja.


In [ ]:
nurseries.tempo.spot_check(2, seed=7)


## 9. O mica analiza

Datele vin gata de folosit. Nimic de mai jos nu tine de pytempo: de aici
incolo e pandas obisnuit.


In [24]:
terr = "Macroregiuni, regiuni de dezvoltare si judete"
cluj = df[(df[terr] == "Cluj") & (df["Sexe"] == "Total")]
cluj = cluj.sort_values("Ani_an")
cluj[[terr, "Ani_an", "Valoare"]].tail(10)

,"Macroregiuni, regiuni de dezvoltare si judete",Ani_an,Valoare
305,Cluj,2015,464.4
306,Cluj,2016,471.0
307,Cluj,2017,468.8
308,Cluj,2018,464.9
309,Cluj,2019,465.5
310,Cluj,2020,467.8
311,Cluj,2021,472.0
312,Cluj,2022,437.1
313,Cluj,2023,443.7
314,Cluj,2024,447.2


Fara grafice aici, intentionat: acest notebook nu adauga nicio dependinta
peste ce ii trebuie deja lui pytempo. Graficele, hartile si modelarea sunt
munca obisnuita pe un DataFrame obisnuit.


In [25]:
cluj["Valoare"].describe()

count     35.000000
mean     456.131429
std       10.760597
min      434.300000
25%      447.450000
50%      459.500000
75%      464.650000
max      472.000000
Name: Valoare, dtype: float64

## 10. Incarcarea in PostgreSQL

pytempo nu se conecteaza niciodata la o baza de date. Scrie SQL-ul ca text,
iar tu decizi cum il rulezi. Asa dependintele raman requests si pandas.

`m.schema()` genereaza CREATE TABLE pentru un indicator: o coloana text per
dimensiune, o valoare numerica, si exact coloanele derivate pe care
`get(tidy=True)` chiar le produce pentru el. Nimic nu se deduce de doua ori,
deci tabelul nu poate diverge de DataFrame.


In [26]:
print(t.matrix("FOM101A").schema())

CREATE TABLE IF NOT EXISTS tempo.fom101a (
    sexe text,
    macroregiuni_regiuni_de_dezvoltare_si_judete text,
    ani text,
    um_mii_persoane text,
    valoare numeric,
    macroregiuni_regiuni_de_dezvoltare_si_judete_nivel text,
    ani_an smallint
);

COMMENT ON TABLE tempo.fom101a IS 'Resurse de munca pe sexe, macroregiuni, regiuni de dezvoltare si judete. Resursele de munca la 1 ianuarie reprezinta acea categorie de populatie care dispune de ansamblul capacitatilor fizice si intelectuale care ii permit sa desfasoare o munca utila in una din activitatile economie nationale';

COMMENT ON COLUMN tempo.fom101a.valoare IS 'Measured in Mii persoane';

CREATE INDEX IF NOT EXISTS fom101a_ani_an_idx ON tempo.fom101a (ani_an);



`t.schema_catalog()` genereaza infrastructura comuna: `indicators` si
`dimensions` descriu catalogul, iar `territory` e un dictionar SIRUTA pe care
il completezi din datele pe care le extragi.


In [27]:
print(t.schema_catalog())

CREATE SCHEMA IF NOT EXISTS tempo;

CREATE TABLE IF NOT EXISTS tempo.indicators (
    code text PRIMARY KEY,
    name text NOT NULL,
    domain text,
    family text,
    periodicity text,
    last_updated text,
    total_cells bigint,
    has_siruta boolean
);

COMMENT ON TABLE tempo.indicators IS 'One row per TEMPO indicator, from the pytempo registry.';

CREATE TABLE IF NOT EXISTS tempo.dimensions (
    code text NOT NULL REFERENCES tempo.indicators (code),
    position smallint NOT NULL,
    label text NOT NULL,
    role text,
    n_options integer,
    PRIMARY KEY (code, position)
);

COMMENT ON TABLE tempo.dimensions IS 'The dimensions of each indicator, in dimensionsMap order.';

CREATE TABLE IF NOT EXISTS tempo.territory (
    siruta integer PRIMARY KEY,
    name text NOT NULL,
    kind text,
    county text
);

COMMENT ON TABLE tempo.territory IS 'SIRUTA lookup, filled from the data you extract.';

CREATE INDEX IF NOT EXISTS territory_county_idx ON tempo.territory (county);



`t.column_mapping(m)` da maparea de la numele coloanelor din DataFrame la
identificatori SQL, deci redenumirea inainte de incarcare e o singura linie.
Fluxul complet e:

    open("catalog.sql", "w").write(t.schema_catalog())
    open("fom101a.sql", "w").write(m.schema())
    # psql -f catalog.sql -f fom101a.sql

    df = df.rename(columns=t.column_mapping(m))
    df.to_sql("fom101a", engine, schema="tempo", if_exists="append",
              index=False)


In [28]:
t.column_mapping(t.matrix("FOM101A"))

{'Sexe': 'sexe',
 'Macroregiuni, regiuni de dezvoltare si judete': 'macroregiuni_regiuni_de_dezvoltare_si_judete',
 'Ani': 'ani',
 'UM: Mii persoane': 'um_mii_persoane',
 'Valoare': 'valoare',
 'Macroregiuni, regiuni de dezvoltare si judete_nivel': 'macroregiuni_regiuni_de_dezvoltare_si_judete_nivel',
 'Ani_an': 'ani_an'}

## Unde afli mai mult

* `t.help()` afiseaza ghidul complet de navigare.
* `m.help()` face acelasi lucru pentru un indicator.
* `m.how()` e meniul unui indicator: fiecare nivel cu marimea si costul lui,
  fiecare dimensiune pe care `select=` o poate ingusta, si un apel tipic de
  copiat.
* `m.territory_columns()` numeste coloanele teritoriului fin, oricum s-ar
  chema dimensiunea la INS.
* `df.tempo.spot_check()` pregateste comparatia cu site-ul.
* README-ul acopera nivelele, rolurile, forma datelor, lipsa fata de zero,
  accessorul de reasezare, puntea spre PostgreSQL si registrul intern de scheme.

Un gand de incheiere. pytempo face o singura treaba: scoate datele din TEMPO,
corect si reproductibil, si ti le da intr-o forma cu care poti lucra. Analiza,
vizualizarea si cartografierea nu sunt treaba lui, si sta deoparte
intentionat, ca sa poti folosi uneltele obisnuite.
